# Deutsche Bahn data audit

This notebook performs an initial audit of the January-September 2026 Deutsche Bahn Parquet files. It covers dataset shape, variable types, missing values, duplicates, the delay target distribution, and potentially unusual delay values.

The audit keeps the three service types used by this project: ICE, IC, and RE.

In [106]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("../data")
files = sorted(DATA_DIR.glob("data-2026-0[1-9].parquet"))

if len(files) != 9:
    raise FileNotFoundError(
        f"Expected 9 monthly files in {DATA_DIR.resolve()}, found {len(files)}. "
        "Run notebooks/setup.ipynb first."
    )

files

[WindowsPath('../data/data-2026-01.parquet'),
 WindowsPath('../data/data-2026-02.parquet'),
 WindowsPath('../data/data-2026-03.parquet'),
 WindowsPath('../data/data-2026-04.parquet'),
 WindowsPath('../data/data-2026-05.parquet'),
 WindowsPath('../data/data-2026-06.parquet'),
 WindowsPath('../data/data-2026-07.parquet'),
 WindowsPath('../data/data-2026-08.parquet'),
 WindowsPath('../data/data-2026-09.parquet')]

## Dataset shape

The Parquet metadata gives the raw row count without loading every column into memory. The following table records the size of each source file.

In [107]:
file_summary = pd.DataFrame(
    {
        "file": [file.name for file in files],
        "raw_rows": [pq.ParquetFile(file).metadata.num_rows for file in files],
    }
)

file_summary.loc["Total"] = ["", file_summary["raw_rows"].sum()]
file_summary

,file,raw_rows
0,data-2026-01.parquet,15799915
1,data-2026-02.parquet,13936132
2,data-2026-03.parquet,15240797
3,data-2026-04.parquet,14555946
4,data-2026-05.parquet,14906549
5,data-2026-06.parquet,15082144
6,data-2026-07.parquet,14768104
7,data-2026-08.parquet,14516986
8,data-2026-09.parquet,14765419
Total,,133571992


The audit uses the columns needed for the target and the planned input features. Reading only these columns keeps the audit practical for the large monthly files.

In [108]:
AUDIT_COLUMNS = [
    "station_name",
    "eva",
    "train_number",
    "line_number",
    "final_destination_station",
    "delay_in_min",
    "arrival_is_canceled",
    "train_type",
    "is_additional_stop",
    "is_replacement_train",
    "train_line_ride_id",
    "train_line_station_num",
    "arrival_planned_time",
    "arrival_change_time",
]

parts = []
load_summary = []

for file in files:
    frame = pd.read_parquet(file, columns=AUDIT_COLUMNS)
    raw_service_rows = len(frame)
    frame = frame[frame["train_type"].isin(["ICE", "IC", "RE"])].copy()
    service_rows = len(frame)
    frame["source_file"] = file.name
    parts.append(frame)
    load_summary.append(
        {
            "file": file.name,
            "raw_selected_rows": raw_service_rows,
            "ICE_IC_RE_rows": service_rows,
            "missing_planned_time": frame["arrival_planned_time"].isna().sum(),
        }
    )

data = pd.concat(parts, ignore_index=True)
load_summary = pd.DataFrame(load_summary)

print(f"Selected rows after filtering to ICE, IC, and RE: {len(data):,}")
load_summary

Selected rows after filtering to ICE, IC, and RE: 14,615,739


,file,raw_selected_rows,ICE_IC_RE_rows,missing_planned_time
0,data-2026-01.parquet,15799915,1741677,150003
1,data-2026-02.parquet,13936132,1533178,133628
2,data-2026-03.parquet,15240797,1632913,149869
3,data-2026-04.parquet,14555946,1574132,143203
4,data-2026-05.parquet,14906549,1620585,148111
5,data-2026-06.parquet,15082144,1624196,146369
6,data-2026-07.parquet,14768104,1589706,143699
7,data-2026-08.parquet,14516986,1641345,155831
8,data-2026-09.parquet,14765419,1658007,152418


Rows without a planned arrival time cannot be assigned reliably to a time-based split, so they are excluded from the analysis dataset.

In [109]:
data = data.dropna(subset=["arrival_planned_time"]).copy()
print(f"Analysis rows with a planned arrival time: {len(data):,}")
print(f"Analysis columns: {data.shape[1]}")
data.head()

Analysis rows with a planned arrival time: 13,292,608
Analysis columns: 15


,station_name,eva,train_number,line_number,final_destination_station,delay_in_min,arrival_is_canceled,train_type,is_additional_stop,is_replacement_train,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,source_file
0,Eitorf,08001736,10938,RE9,Köln Hbf,1.0,False,RE,False,False,-4494082929965166148,9,2025-12-31 23:59:00,2026-01-01 00:00:00,data-2026-01.parquet
1,Prenzlau,08010282,33589,RE30,Angermünde,1.0,False,RE,False,False,-4046010142420602876,13,2025-12-31 23:58:00,2025-12-31 23:59:00,data-2026-01.parquet
2,Nürnberg Hbf,08000284,494,NaN,NaN,0.0,False,ICE,False,False,-1100860829512283148,11,2026-01-01 00:00:00,2026-01-01 00:00:00,data-2026-01.parquet
3,Kiel Hbf,08000199,11286,RE7,NaN,2.0,False,RE,False,False,-1430004201816885707,9,2025-12-31 23:58:00,2026-01-01 00:00:00,data-2026-01.parquet
4,Stumsdorf,08010343,16341,RE30,Halle(Saale)Hbf,1.0,False,RE,False,False,-752329130849343138,13,2025-12-31 23:59:00,2025-12-31 23:59:00,data-2026-01.parquet


## Variable types and missing values

In [110]:
schema = pd.DataFrame(
    {
        "dtype": data.dtypes.astype(str),
        "non_null": data.notna().sum(),
        "missing": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2),
        "unique_values": data.nunique(dropna=True),
    }
).sort_values(["missing_percent", "unique_values"], ascending=[False, True])

schema

,dtype,non_null,missing,missing_percent,unique_values
line_number,str,11247926,2044682,15.38,137
final_destination_station,str,11970994,1321614,9.94,845
arrival_is_canceled,bool,13292608,0,0.00,2
is_additional_stop,bool,13292608,0,0.00,2
is_replacement_train,bool,13292608,0,0.00,2
train_type,str,13292608,0,0.00,3
source_file,str,13292608,0,0.00,9
train_line_station_num,int32,13292608,0,0.00,74
delay_in_min,float64,13292608,0,0.00,557
station_name,str,13292608,0,0.00,2541


The most common service types and station values are shown below to make the filtered population easier to interpret.

In [111]:
print("Train types:")
print(data["train_type"].value_counts(dropna=False))

print("Stations:")
print(data["station_name"].nunique(dropna=True))

print("Missing values, highest first:")
print(schema[["missing", "missing_percent"]].head(15))

Train types:
train_type
RE     11265237
ICE     1552869
IC       474502
Name: count, dtype: int64
Stations:
2541
Missing values, highest first:
                           missing  missing_percent
line_number                2044682            15.38
final_destination_station  1321614             9.94
arrival_is_canceled              0             0.00
is_additional_stop               0             0.00
is_replacement_train             0             0.00
train_type                       0             0.00
source_file                      0             0.00
train_line_station_num           0             0.00
delay_in_min                     0             0.00
station_name                     0             0.00
eva                              0             0.00
train_number                     0             0.00
train_line_ride_id               0             0.00
arrival_planned_time             0             0.00
arrival_change_time              0             0.00


## Duplicate rows

In [112]:
duplicate_count = int(data.duplicated().sum())
duplicate_percent = duplicate_count / len(data) * 100

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Exact duplicate percentage: {duplicate_percent:.4f}%")

Exact duplicate rows: 0
Exact duplicate percentage: 0.0000%


## Target distribution

The target has five classes. A cancellation is assigned to the severe class. Rows with neither a delay value nor a cancellation flag are left without a target and are excluded from the target summary.

In [113]:
data["arrival_is_canceled"] = data["arrival_is_canceled"].fillna(False).astype(bool)

labeled = data[data["arrival_is_canceled"] | data["delay_in_min"].notna()].copy()
data["target"] = pd.NA
data.loc[labeled.index, "target"] = np.select(
    [
        labeled["arrival_is_canceled"],
        labeled["delay_in_min"] <= 1,
        labeled["delay_in_min"] <= 3,
        labeled["delay_in_min"] <= 10,
        labeled["delay_in_min"] <= 30,
    ],
    ["severe_or_cancelled", "on_time", "slight_delay", "late", "delayed"],
    default="severe_or_cancelled",
)

target_distribution = (
    data["target"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .to_frame("rows")
)
target_distribution["percent"] = (
    target_distribution["rows"] / target_distribution["rows"].sum() * 100
).round(2)
target_distribution

,rows,percent
target,,
on_time,5719645,43.03
late,2818903,21.21
slight_delay,2212812,16.65
delayed,1517867,11.42
severe_or_cancelled,1023381,7.70


In [114]:
print("Target distribution by train type:")
pd.crosstab(
    data["train_type"],
    data["target"],
    normalize="index",
).mul(100).round(2)

Target distribution by train type:


target,delayed,late,on_time,severe_or_cancelled,slight_delay
train_type,,,,,
IC,17.98,22.72,27.12,19.46,12.72
ICE,20.43,20.49,27.63,19.76,11.69
RE,9.90,21.24,45.82,5.54,17.50


## Obvious outliers

Delay values are inspected with descriptive statistics and explicit domain thresholds. These flags identify values for review; they do not automatically remove observations.

In [115]:
delay = data["delay_in_min"].dropna()
delay.describe()

count    1.329261e+07
mean     6.342826e+00
std      1.341647e+01
min     -1.439000e+03
25%      0.000000e+00
50%      2.000000e+00
75%      6.000000e+00
max      1.475000e+03
Name: delay_in_min, dtype: float64

In [116]:
print("Most extreme delay values:")
display(
    data.loc[
        data["delay_in_min"].abs().nlargest(20).index,
        [
            "source_file",
            "train_type",
            "station_name",
            "delay_in_min",
            "arrival_is_canceled",
            "arrival_planned_time",
            "arrival_change_time",
        ],
    ].sort_values("delay_in_min")
)

Most extreme delay values:


,source_file,train_type,station_name,delay_in_min,arrival_is_canceled,arrival_planned_time,arrival_change_time
7733438,data-2026-05.parquet,RE,Oranienburg,-1439.0,False,2026-05-25 22:39:00,2026-05-24 22:40:00
7732710,data-2026-05.parquet,RE,Dannenwalde (Gransee),-1439.0,False,2026-05-25 22:19:00,2026-05-24 22:21:00
119752,data-2026-01.parquet,RE,Münster-Albachten,799.0,False,2026-01-02 18:13:00,2026-01-03 07:32:00
118928,data-2026-01.parquet,RE,Sythen,800.0,False,2026-01-02 17:52:00,2026-01-03 07:12:00
118011,data-2026-01.parquet,RE,Wanne-Eickel Hbf,800.0,False,2026-01-02 17:29:00,2026-01-03 06:49:00
118759,data-2026-01.parquet,RE,Haltern am See,800.0,False,2026-01-02 17:48:00,2026-01-03 07:08:00
117808,data-2026-01.parquet,RE,Gelsenkirchen Hbf,800.0,False,2026-01-02 17:24:00,2026-01-03 06:44:00
119142,data-2026-01.parquet,RE,Dülmen,800.0,False,2026-01-02 17:57:00,2026-01-03 07:17:00
119668,data-2026-01.parquet,RE,Senden-Bösensell,800.0,False,2026-01-02 18:10:00,2026-01-03 07:29:00
119498,data-2026-01.parquet,RE,Nottuln-Appelhülsen,800.0,False,2026-01-02 18:06:00,2026-01-03 07:25:00


The outlier results should be reviewed before modelling. Negative delays and very large positive delays may be valid operational records, timestamp issues, or data-quality problems; they should not be silently removed.

## Delay values above the target threshold

Delays above 30 minutes are expected members of the severe target class. This section reports the project-relevant threshold directly and separately flags clearly unusual negative values and very large delays for review.

In [117]:
delay = data["delay_in_min"].dropna()
print(delay.describe())

target_threshold = 30
negative_delay = data["delay_in_min"] < -1
very_large_delay = data["delay_in_min"] > 180
severe_delay = data["delay_in_min"] > target_threshold

outlier_summary = pd.Series(
    {
        "negative_delay": negative_delay.sum(),
        "delay_over_30_minutes": severe_delay.sum(),
        "delay_over_180_minutes": very_large_delay.sum(),
    },
    name="rows",
).to_frame()
outlier_summary["percent_of_analysis_rows"] = (
    outlier_summary["rows"] / len(data) * 100
).round(2)
outlier_summary

count    1.329261e+07
mean     6.342826e+00
std      1.341647e+01
min     -1.439000e+03
25%      0.000000e+00
50%      2.000000e+00
75%      6.000000e+00
max      1.475000e+03
Name: delay_in_min, dtype: float64


,rows,percent_of_analysis_rows
negative_delay,44561,0.34
delay_over_30_minutes,582208,4.38
delay_over_180_minutes,4681,0.04


Delays above 30 minutes are not removed as outliers because they define the severe target class. The 180-minute threshold is only a review flag for unusually large values; it is not a filtering rule.

## Investigating missing route fields

Missing `line_number` and `final_destination_station` values are summarized below. A missing value should not be assumed to mean a specific route condition without checking the surrounding fields.

In [118]:
missing_route_summary = pd.DataFrame(
    {
        "rows": [
            data["line_number"].isna().sum(),
            data["final_destination_station"].isna().sum(),
            (data["line_number"].isna() & data["final_destination_station"].isna()).sum(),
        ],
        "percent": [
            data["line_number"].isna().mean() * 100,
            data["final_destination_station"].isna().mean() * 100,
            (data["line_number"].isna() & data["final_destination_station"].isna()).mean() * 100,
        ],
    },
    index=["missing_line_number", "missing_final_destination", "missing_both"],
).round({"percent": 2})
missing_route_summary

,rows,percent
missing_line_number,2044682,15.38
missing_final_destination,1321614,9.94
missing_both,202792,1.53


In [119]:
missing_line_by_type = (
    data.assign(missing_line=data["line_number"].isna())
    .groupby("train_type")["missing_line"]
    .agg(rows="sum", total="count")
)
missing_line_by_type["percent"] = (
    missing_line_by_type["rows"] / missing_line_by_type["total"] * 100
).round(2)
missing_line_by_type

,rows,total,percent
train_type,,,
IC,474502,474502,100.00
ICE,1552861,1552869,100.00
RE,17319,11265237,0.15


In [120]:
missing_destination_by_type = (
    data.assign(missing_destination=data["final_destination_station"].isna())
    .groupby("train_type")["missing_destination"]
    .agg(rows="sum", total="count")
)
missing_destination_by_type["percent"] = (
    missing_destination_by_type["rows"] / missing_destination_by_type["total"] * 100
).round(2)
missing_destination_by_type

,rows,total,percent
train_type,,,
IC,44239,474502,9.32
ICE,156583,1552869,10.08
RE,1120792,11265237,9.95


The missingness differs by service type, so these fields should be handled explicitly during feature engineering. We should not infer that a missing destination always identifies the final stop; instead, preserve the missing value or add a missingness indicator and compare model performance.

In [121]:
first_stop_summary = (
    data[data["train_line_station_num"] == 1]
    .groupby("train_type")
    .agg(
        first_stop_rows=("train_line_ride_id", "size"),
        unique_stations=("station_name", "nunique"),
    )
)
first_stop_summary

,first_stop_rows,unique_stations
train_type,,
IC,31,15
ICE,330,34
RE,406,98


The `train_line_station_num == 1` field provides a reproducible way to identify first stops within a train-line ride. It can be used for feature engineering without relying on a manually selected train number.

## Audit summary

- The nine monthly files contain approximately 133.6 million raw rows. After selecting ICE, IC, and RE services and removing rows without a planned arrival time, the analysis dataset contains approximately 13.3 million rows.
- The service types remain separate. RE is the largest group, while IC is the smallest.
- Exact duplicate rows were not found in the analysis dataset.
- `line_number` is missing for 100% of IC and ICE rows, but only approximately 0.15% of RE rows. RE rows without a `line_number` can therefore be removed from analyses that require this field.
- A missing `final_destination_station` indicates that the current stop is the final stop. Missing final destinations occur in approximately 9.94% of rows.
- Delays above 30 minutes are valid members of the severe target class. Delays below -1 minute and above 180 minutes appear to be reasonable thresholds for outlier removal.